# AIFM Private Debt Fund

This notebook presents a private debt risk-monitoring workflow for a simulated AIF portfolio. The fund invests in senior secured loans, high-yield bonds, and CLO exposures.

The analysis focuses on credit and portfolio risk indicators, including credit quality, covenant headroom, maturity profile, leverage, liquidity, borrower default stress, investor concentration, and selected sustainability indicators. Regulatory context is AIFMD-style fund governance, but the notebook focuses on the monitoring workflow and interpretation of the resulting risk metrics.

> **Output gallery:** Tables and plots generated by this notebook are saved in the [`figs/aifm_private_debt`](../../figs/aifm_private_debt) folder. Readers who prefer to review the generated outputs directly can browse that folder without running the notebook.

> For supporting methodology notes, see [Private Debt Notes](../../docs/notebooks_notes/private_debt.md).

In [ ]:
# from src.config import VALUATION_DATE, QUARTER
# from src.risk.reg_constants import CONFIDENCE, HORIZON, NOTICE, GROSS_LIMIT

# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from src.ui.plot_style import (
#     C,  FUND_COLORS, ACCENT, ACCENT2, ACCENT3, WARNING,
#     apply_ax_style, section_title, 
# )
# from src.ui.nb_utils import save_fig, save_table, styled_table, save_table_png

# import warnings
# warnings.filterwarnings('ignore')

# from src.setup_db import run as setup_db
# setup_db()

# from src.data.database import get_engine, query_positions, query_nav_history
# from src.data.enrichment import get_risk_ready_df
# from src.data.mock_bloomberg import MockBloomberg as Bloomberg
# from src.risk.leverage_config import INSTRUMENT_SOURCE
# from src.risk.risk_utils import (
#     var_historical, var_parametric, var_scale,
#     es_historical, es_scale,
#     exception_report, full_backtest_report,
#     stress_equity, stress_rates, stress_credit,
#     stress_fx, stress_combined, stress_historical,
#     days_to_liquidate, liquidity_buckets, redemption_stress, investor_concentration,
#     liquidity_adjusted_var,
# )
# import src.risk.risk_utils as rk
# import src.ui.print_html_utils as phtml
# import src.print_utils as prt
# import src.risk.esg_utils as esg_u 
# from src.risk.esg_utils import ESG_FIELDS, ESG_THRESHOLD_LOW
# from src.ui.nb_utils import save_fig, save_table, styled_table

# FUND_ID    = 'AIFM_PrivateDebt'
# VALUATION_DATE      = '2026-05-13'
# ENGINE     = get_engine()
# BBG        = Bloomberg()

# HORIZON    = 20

In [ ]:
import warnings
warnings.filterwarnings('ignore')

# to initialize db if no tables exist abd to create session factory
from src.setup_db import run as setup_db
setup_db()
from src.data.database import get_engine
ENGINE = get_engine()

# initialize mock bloomberg API
from src.data.mock_bloomberg import MockBloomberg as Bloomberg
BBG = Bloomberg()

# helper functions for risk computations and nice output formatting
import src.risk.risk_utils as rk
import src.ui.print_html_utils as phtml

### Fund Example in this Notebook

The fund profile below sets the operating context for the risk workflow, including strategy, fund type, base currency, reporting setup, and monitoring framework.

In [ ]:
# Display fund overview banner — fund identity and risk methodology framework
FUND_ID    = 'AIFM_PrivateDebt'
phtml.display_fund_overview_banner(
    fund_id=FUND_ID,
    engine=ENGINE,
    export_id="01",
)

> Note: Fund characteristics, risk limits, methodologies, and reporting parameters are simulated.

---

### Risk Management Policy Parameters

The fund's risk parameters are sourced from the Risk Management Policy configuration and used throughout the notebook for measurement, monitoring, and limit checks.

In [ ]:
# Display Risk Management Policy parameters from fund reference data
phtml.display_fund_rmp_parameters(
    fund_id=FUND_ID,
    engine=ENGINE,
)

The RMP is loaded as `rmp` and passed to the relevant risk functions. This keeps fund-specific parameters outside the calculation code.

In [ ]:
# read in RMP parameters for the fund
from src.data.reference_data import load_rmp
rmp = load_rmp(FUND_ID)

### Implementation Context

The analysis is performed as of a fixed valuation date, consistent with the point-in-time design used across the fund workflows.

In [ ]:
# fixed valuation date for all computations in this notebook
from src.config import VALUATION_DATE
VALUATION_DATE

Portfolio positions, fund characteristics, counterparty records, reference data, and market data are retrieved from the SQLite data layer. Market data is enriched through the simulated Bloomberg workflow before being passed to the risk analytics modules.

For a fuller explanation of the data workflow, see the [Data Layer Workflow](../data_workflows/01_data_layer_workflow.ipynb).

In [ ]:
# enrich fund postion data w/ info from Bloomberg
from src.data.enrichment import get_risk_ready_df
risk_df = get_risk_ready_df(ENGINE, FUND_ID, VALUATION_DATE)

# display the enriched risk dataframe first 2 rows
risk_df.head(2)

From this point onward, the notebook uses helper functions to render summary tables and HTML views. Data aggregation, calculations, and reporting logic remain inside the package modules.

In [ ]:
# query from SQLdb
from src.data.database import query_positions
positions = query_positions(ENGINE, FUND_ID, VALUATION_DATE)
NAV = risk_df['market_value_eur'].sum()

phtml.display_fund_summary(FUND_ID, VALUATION_DATE, positions, risk_df, NAV, export_id="03")

In [ ]:
phtml.display_top_positions(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="04")

In [ ]:
phtml.display_asset_class_breakdown(risk_df, valuation_date=VALUATION_DATE, fund_id=FUND_ID, export_id="05")  # Also computes NAV internally

---

## 4. Leverage Monitoring

The notebook reports gross and commitment-style leverage measures and compares the results with fund-level monitoring limits.

For this private debt fund, the leverage output supports Annex IV-style reporting inputs and helps distinguish borrowing, portfolio exposure, and synthetic exposure where applicable.

In [ ]:
# MRS-23: Leverage computation - Gross and Commitment method

# ----------------------------------------------------------------
# Gross method: sum of absolute exposures / NAV
# ----------------------------------------------------------------
risk_df['abs_exposure'] = risk_df['market_value_eur'].abs()

deriv_rows     = risk_df[risk_df['asset_class'] == 'Derivative'].copy()
deriv_notional = 0.0
for _, row in deriv_rows.iterrows():
    ticker        = 'SPXW 260619P05500 Index'
    bbg_data      = BBG.bdp(ticker, ['DELTA', 'OPT_UNDL_PX', 'CONTRACT_SIZE'])
    delta         = abs(bbg_data.loc[ticker, 'DELTA'])
    undl_px       = bbg_data.loc[ticker, 'OPT_UNDL_PX']
    contract_size = bbg_data.loc[ticker, 'CONTRACT_SIZE']
    quantity      = abs(row['quantity'])
    fx_rate       = 0.89
    deriv_notional += delta * quantity * contract_size * undl_px * fx_rate

risk_df['gross_exposure'] = risk_df.apply(
    lambda r: deriv_notional if r['asset_class'] == 'Derivative'
    else (0.0 if r['asset_class'] == 'Cash'
    else r['abs_exposure']),
    axis=1
)

gross_leverage = risk_df['gross_exposure'].sum() / NAV

# ----------------------------------------------------------------
# Commitment method
# ----------------------------------------------------------------
long_eq  = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] > 0)]['market_value_eur'].sum()
short_eq = risk_df[(risk_df['asset_class'] == 'Equity') &
                   (risk_df['market_value_eur'] < 0)]['market_value_eur'].sum()
net_eq   = abs(long_eq + short_eq)
bonds    = risk_df[risk_df['asset_class'].isin(['Bond','Loan','CLO'])]['market_value_eur'].abs().sum()
fx       = risk_df[risk_df['asset_class'] == 'FX']['market_value_eur'].abs().sum()

commitment_exposure = net_eq + bonds + fx + deriv_notional
commitment_leverage = commitment_exposure / NAV

# ----------------------------------------------------------------
# Summary table
# ----------------------------------------------------------------
all_classes = sorted(risk_df['asset_class'].unique())

leverage_summary = pd.DataFrame({
    'Gross (EUR)'        : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()
                            for ac in all_classes],
    'Gross (x NAV)'      : [risk_df[risk_df['asset_class']==ac]['gross_exposure'].sum()/NAV
                            for ac in all_classes],
    'Commitment (EUR)'   : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional if ac == 'Derivative' else 0)
                            for ac in all_classes],
    'Commitment (x NAV)' : [risk_df[risk_df['asset_class']==ac]['market_value_eur'].abs().sum()/NAV
                            if ac not in ['Cash', 'Derivative'] else
                            (deriv_notional/NAV if ac == 'Derivative' else 0)
                            for ac in all_classes],
}, index=all_classes)

leverage_summary['Gross (EUR)']        = leverage_summary['Gross (EUR)'].map('{:,.0f}'.format)
leverage_summary['Gross (x NAV)']      = leverage_summary['Gross (x NAV)'].map('{:.2f}x'.format)
leverage_summary['Commitment (EUR)']   = leverage_summary['Commitment (EUR)'].map('{:,.0f}'.format)
leverage_summary['Commitment (x NAV)'] = leverage_summary['Commitment (x NAV)'].map('{:.2f}x'.format)

print(f"{'Asset Class':<15} {'Gross (EUR)':>15} {'Gross':>8} {'Commit (EUR)':>15} {'Commit':>8}")
print('-' * 65)
for ac in all_classes:
    row = leverage_summary.loc[ac]
    print(f"{ac:<15} {row['Gross (EUR)']:>15} {row['Gross (x NAV)']:>8} "
          f"{row['Commitment (EUR)']:>15} {row['Commitment (x NAV)']:>8}")
print('-' * 65)
print(f"{'Total':<15} {risk_df['gross_exposure'].sum():>15,.0f} {gross_leverage:>7.2f}x "
      f"{commitment_exposure:>15,.0f} {commitment_leverage:>7.2f}x")

GROSS_LIMIT = 3.0
status      = 'OK' if gross_leverage <= GROSS_LIMIT else 'BREACH'
print(f"\nGross leverage limit : {GROSS_LIMIT:.0f}x")
print(f"Current gross        : {gross_leverage:.2f}x")
print(f"Status               : {status}")

In [ ]:
# AIFMD II granular leverage breakdown
granular = risk_df.groupby(['asset_class', 'sub_asset_class']).agg(
    gross_eur=('gross_exposure', 'sum'),
    n_positions=('isin', 'count')
).reset_index()
granular['gross_x_nav'] = granular['gross_eur'] / NAV
granular['source']      = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[0], axis=1)
granular['listed_otc']  = granular.apply(
    lambda r: INSTRUMENT_SOURCE.get(
        (r['asset_class'], r['sub_asset_class']), ('Other', 'Other'))[1], axis=1)

granular = granular[granular['gross_eur'] > 0].sort_values('gross_eur', ascending=False)

# listed vs OTC summary
total_gross = granular['gross_eur'].sum()
summary_lot = granular.groupby('listed_otc')['gross_eur'].sum().reset_index()
summary_lot['x_nav']        = summary_lot['gross_eur'] / NAV
summary_lot['pct_leverage'] = summary_lot['gross_eur'] / total_gross * 100
summary_lot['gross_eur']    = summary_lot['gross_eur'].map('{:,.0f}'.format)
summary_lot['x_nav']        = summary_lot['x_nav'].map('{:.2f}x'.format)
summary_lot['pct_leverage'] = summary_lot['pct_leverage'].map('{:.1f}%'.format)
summary_lot.index.name      = None
summary_lot.columns         = ['Category', 'Gross (EUR)', 'x NAV', '% Leverage']
summary_lot.set_index('Category', inplace=True)

header = f"{'':12} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_lot.iterrows():
    print(f"{idx:<12} {row['Gross (EUR)']:>15} {row['x NAV']:>8} {row['% Leverage']:>12}")
print('-' * len(header))
print()

summary_src = granular.groupby('source')['gross_eur'].sum().reset_index()
summary_src['x_nav']        = summary_src['gross_eur'] / NAV
summary_src['pct_leverage'] = summary_src['gross_eur'] / total_gross * 100
summary_src['gross_eur']    = summary_src['gross_eur'].map('{:,.0f}'.format)
summary_src['x_nav']        = summary_src['x_nav'].map('{:.2f}x'.format)
summary_src['pct_leverage'] = summary_src['pct_leverage'].map('{:.1f}%'.format)
summary_src.set_index('source', inplace=True)
summary_src.index.name      = None

header = f"{'':20} {'Gross (EUR)':>15} {'x NAV':>8} {'% Leverage':>12}"
print(header)
print('-' * len(header))
for idx, row in summary_src.iterrows():
    print(f"{idx:<20} {row['gross_eur']:>15} {row['x_nav']:>8} {row['pct_leverage']:>12}")
print('-' * len(header))
print()

# granular table
granular['pct_leverage'] = (granular['gross_eur'] / total_gross * 100).map('{:.1f}%'.format)
granular['gross_eur']    = granular['gross_eur'].map('{:,.0f}'.format)
granular['gross_x_nav']  = granular['gross_x_nav'].map('{:.2f}x'.format)
granular.set_index(['source', 'asset_class', 'sub_asset_class'], inplace=True)
granular

---

## 5. Stress Testing

The notebook applies rate, credit spread, combined, and historical stress scenarios to the private debt portfolio.

```text
Stress P&L = sensitivity × shock × market value
```

The output shows the portfolio impact of rate and credit stresses across senior loans, high-yield bonds, and CLO exposures.

In [ ]:
# MRS-26: Annex VI stress scenarios - private debt
from src.risk.risk_utils import HISTORICAL_SCENARIOS

# scenario table
rows = []
for key, p in HISTORICAL_SCENARIOS.items():
    rows.append({
        'Scenario'    : p['name'],
        'Equity'      : f"{p['delta_equity']*100:.0f}%",
        'Rates (bps)' : f"{p['delta_y']*10000:.0f}",
        'Credit (bps)': f"+{p['delta_spread']*10000:.0f}",
        'USD'         : f"{p['fx_shocks'].get('USD', 0)*100:+.0f}%",
        'GBP'         : f"{p['fx_shocks'].get('GBP', 0)*100:+.0f}%",
    })
pd.DataFrame(rows).set_index('Scenario')

In [ ]:
# stress results
rt  = stress_rates(risk_df, delta_y=0.02)
cr  = stress_credit(risk_df, delta_spread=0.015)
cb  = stress_combined(risk_df)
hist = {s: stress_historical(risk_df, s) for s in HISTORICAL_SCENARIOS}

rows = [
    {'Scenario': 'Rate Shock +200bps',      'P&L (EUR)': rt['stressed_pnl_eur'], '% NAV': rt['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Credit Widening +150bps', 'P&L (EUR)': cr['stressed_pnl_eur'], '% NAV': cr['stressed_pnl_eur']/NAV*100},
    {'Scenario': 'Combined',                'P&L (EUR)': cb['stressed_pnl_eur'], '% NAV': cb['stressed_pnl_eur']/NAV*100},
] + [
    {'Scenario': v['scenario'], 'P&L (EUR)': v['stressed_pnl_eur'], '% NAV': v['stressed_pnl_eur']/NAV*100}
    for v in hist.values()
]

summary_raw = pd.DataFrame(rows).set_index('Scenario')
worst_idx   = summary_raw['% NAV'].idxmin()

summary_raw['P&L (EUR)'] = summary_raw['P&L (EUR)'].map('{:,.0f}'.format)
summary_raw['% NAV']     = summary_raw['% NAV'].map('{:.2f}%'.format)

summary_raw.style.apply(lambda x: [
    'background-color: #7f1d1d; color: white' if i == worst_idx else ''
    for i in x.index], axis=0)

---

## 6. Liquidity Profile

The notebook groups portfolio assets by estimated time to liquidation and compares liquidity capacity with the fund's monitoring assumptions.

```text
Days to liquidate = absolute market value / (average daily volume × participation rate)
```

Cash and near-cash instruments are treated as the most liquid assets. Instruments with no usable trading volume are treated as illiquid.

In [ ]:
# MRS-30: Liquidity profile
from src.config import LIQUIDITY_BUCKET_ORDER
from src.risk.risk_utils import days_to_liquidate, liquidity_buckets, redemption_stress

risk_df_liq = days_to_liquidate(risk_df, pct_adv=0.25)
risk_df_liq = liquidity_buckets(risk_df_liq)

bucket_summary = risk_df_liq.groupby('liquidity_bucket').agg(
    market_value_eur=('market_value_eur', 'sum'),
    abs_exposure=('market_value_eur', lambda x: x.abs().sum()),
    n_positions=('isin', 'count')
).reset_index()
bucket_summary['pct_nav_net'] = bucket_summary['market_value_eur'] / NAV * 100
bucket_summary['pct_nav_abs'] = bucket_summary['abs_exposure'] / NAV * 100

bucket_full = bucket_summary.set_index('liquidity_bucket').reindex(LIQUIDITY_BUCKET_ORDER).fillna(0).reset_index()

# table
print(f"{'Bucket':<15} {'Abs Exposure (EUR)':>20} {'% NAV':>8} {'# Pos':>6}")
print('-' * 55)
for _, row in bucket_full.iterrows():
    if row['abs_exposure'] > 0:
        print(f"{row['liquidity_bucket']:<15} {row['abs_exposure']:>20,.0f} "
              f"{row['pct_nav_abs']:>7.1f}% {row['n_positions']:>6.0f}")
    else:
        print(f"{row['liquidity_bucket']:<15} {'—':>20} {'—':>8} {'—':>6}")
print('-' * 55)
total_abs = risk_df_liq['market_value_eur'].abs().sum()
print(f"{'Total':<15} {total_abs:>20,.0f} {total_abs/NAV*100:>7.1f}%")


# chart
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(bucket_full['liquidity_bucket'],
              bucket_full['pct_nav_abs'],
              color=ACCENT, alpha=0.85, width=0.5)
ax.axhline(0, color=C['dim'], lw=0.8)
ax.set_ylabel('Absolute Exposure (% NAV)', fontsize=9)
section_title(ax, "Liquidity Profile — Absolute Exposure by Bucket")

for bar, val in zip(bars, bucket_full['pct_nav_abs']):
    if val > 2:
        ax.text(bar.get_x() + bar.get_width()/2, val/2,
                f'{val:.1f}%', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')
plt.tight_layout()
save_fig(fig, FUND_ID, "Liquidity buckets PE")
plt.show()

---

### 6.2 Redemption Stress

Redemption stress compares liquid assets available within the contractual notice period against simulated redemption amounts.

The notebook tests normal, large, stressed, and largest-investor redemption scenarios.

In [ ]:
# MRS-31: Redemption stress — AIFM Private Debt
NOTICE = 5   # contractual notice period (days)
_SCENARIOS = [
    (0.10, 'Normal    (10% NAV)'),
    (0.25, 'Large     (25% NAV)'),
    (0.50, 'Stress    (50% NAV)'),
]

print(f'Fund: AIFM Private Debt  |  NAV: EUR {NAV:,.0f}  |  Notice: {NOTICE} days')
print()
print(f"{'':22} {'Redemption (M)':>14} {'Liquid (M)':>12} {'Gap (M)':>12} {'Coverage':>10} Action")
print('\u2500' * 95)

_redstress = {}
for _pct, _label in _SCENARIOS:
    _r = redemption_stress(risk_df_liq, NAV, redemption_pct=_pct, notice_days=NOTICE)
    _redstress[_pct] = _r
    _gap = f"+{_r['liquidity_gap_eur']/1e6:.1f}M" if _r['liquidity_gap_eur'] >= 0 else f"{_r['liquidity_gap_eur']/1e6:.1f}M"
    print(f"{_label:<22} {_r['redemption_amount_eur']/1e6:>13.1f}M {_r['liquid_assets_eur']/1e6:>11.1f}M "
          f"{_gap:>12} {_r['coverage_ratio']:>9.2f}x  {_r['recommendation']}")
print('\u2500' * 95)
print('Largest-investor stress: see Section 6.3')

### 6.3 Investor Concentration

The notebook monitors investor concentration and uses the investor register to derive the largest-investor redemption stress scenario.

The output links investor concentration with potential liquidity pressure.

In [ ]:
# MRS-32: Investor concentration — AIFM Private Debt
_investors = pd.DataFrame([
    {'investor_id': 'PD001', 'investor_name': 'Nordic Sovereign Wealth Fund', 'investor_type': 'Sovereign Wealth', 'aum_eur': NAV * 0.35}, 
    {'investor_id': 'PD002', 'investor_name': 'European Insurance Alliance', 'investor_type': 'Insurance', 'aum_eur': NAV * 0.20}, 
    {'investor_id': 'PD003', 'investor_name': 'German Pension Consortium', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.15}, 
    {'investor_id': 'PD004', 'investor_name': 'Danish Pension Fund A', 'investor_type': 'Pension Fund', 'aum_eur': NAV * 0.06}, 
    {'investor_id': 'PD005', 'investor_name': 'Institutional Investor B', 'investor_type': 'Asset Manager', 'aum_eur': NAV * 0.06}, 
    {'investor_id': 'PD006', 'investor_name': 'Insurance Holding C', 'investor_type': 'Insurance', 'aum_eur': NAV * 0.06}, 
    {'investor_id': 'PD007', 'investor_name': 'Family Office D', 'investor_type': 'Family Office', 'aum_eur': NAV * 0.06}, 
    {'investor_id': 'PD008', 'investor_name': 'Asset Manager E', 'investor_type': 'Asset Manager', 'aum_eur': NAV * 0.06}, 
])

_conc = investor_concentration(_investors, NAV, threshold=0.20)
_top  = _conc['by_investor']
_type = _investors.set_index('investor_id')['investor_type']

print(f'Investor Concentration — AIFM Private Debt  |  NAV: EUR {NAV:,.0f}')
print('ESMA threshold: 20% single / 50% top-3\n')
print(f"{'':4} {'Investor':<30} {'Type':<18} {'AUM (EUR)':>14} {'% NAV':>8}")
print('\u2500' * 80)
for _rank, (_, _row) in enumerate(_top.iterrows(), 1):
    _t = _type.get(_row['investor_id'], '')
    print(f"{_rank:<4} {_row['investor_name']:<30} {_t:<18} {_row['aum_eur']:>14,.0f} {_row['pct_nav']*100:>7.1f}%")
print('\u2500' * 80)

_flag_s = '\u26a0 ESMA flag'       if _conc['concentration_flag'] else '\u2713 OK'
_flag_3 = '\u26a0 High conc.'      if _conc['high_concentration'] else '\u2713 OK'
print(f"\nLargest investor : {_conc['largest_investor_pct']*100:.1f}% NAV  {_flag_s}")
print(f"Top 3 investors  : {_conc['top3_pct']*100:.1f}% NAV  {_flag_3}")

# Largest-investor redemption stress (4th scenario)
_r4   = redemption_stress(risk_df_liq, NAV, redemption_pct=_conc['largest_investor_pct'], notice_days=5)
_gap4 = f"+{_r4['liquidity_gap_eur']/1e6:.1f}M" if _r4['liquidity_gap_eur'] >= 0 else f"{_r4['liquidity_gap_eur']/1e6:.1f}M"
print(f"\nLargest-investor stress ({_conc['largest_investor_pct']*100:.1f}% NAV, 5-day notice):")
print(f"  Redemption : EUR {_r4['redemption_amount_eur']:,.0f}")
print(f"  Liquid     : EUR {_r4['liquid_assets_eur']:,.0f}")
print(f"  Gap        : {_gap4}  |  Coverage: {_r4['coverage_ratio']:.2f}x")
print(f"  Action     : {_r4['recommendation']}")

print('\nMonitoring recommendation:')
if _conc['high_concentration']:
    print('  \u2014 Enhanced monitoring: top-3 investors represent significant co-ordinated exit risk')
    print('  \u2014 Maintain liquidity buffer >= largest investor AUM')
if _conc['concentration_flag']:
    print(f'  \u2014 Gate-trigger review: largest investor at {_conc["largest_investor_pct"]*100:.1f}% NAV')
if not _conc['concentration_flag'] and not _conc['high_concentration']:
    print('  \u2014 No immediate action. Continue quarterly investor concentration monitoring.')

### 6.4 Counterparty Stress

For a private debt fund, borrower default is a main counterparty risk.

The notebook models the largest single borrower defaulting on its outstanding principal and applies a recovery assumption to estimate the loss impact.

In [ ]:
import re

def extract_issuer(name):
    # remove CLO/ICS/fund suffixes and ratings/years
    name = re.sub(r'\b(CLO|ICS|AAA|AA\b|BBB|[A-Z]{1,3}\d+)\b.*', '', name)
    # remove coupon and maturity patterns like 6.5 2028
    name = re.sub(r'\s+\d+\.\d+\s+\d{4}.*', '', name)
    # remove standalone years
    name = re.sub(r'\s+\d{4}.*', '', name)
    return name.strip()

In [ ]:
# MRS-35: Counterparty stress — AIFM Private Debt
# Derive largest borrower exposure from live risk_df
_RECOVERY_RATE = 0.40  # senior secured LGD ~60%, ESMA benchmark

_loans_pd = risk_df[risk_df['market_value_eur'] > 0].copy()
_loans_pd ['issuer'] = _loans_pd ['instrument_name'].apply(extract_issuer)
_by_borrower = (
    _loans_pd
    .groupby('issuer', dropna=True)['market_value_eur']
    .sum()
    .reset_index()
    .rename(columns={'market_value_eur': 'exposure_eur'})
    .sort_values('exposure_eur', ascending=False)
    .reset_index(drop=True)
)
_by_borrower['loss_eur'] = _by_borrower['exposure_eur'] * (1 - _RECOVERY_RATE)
_by_borrower['exp_pct']  = _by_borrower['exposure_eur'] / NAV
_by_borrower['loss_pct'] = _by_borrower['loss_eur'] / NAV

_worst_borrow    = _by_borrower.iloc[0]
_borrow_loss_eur = _worst_borrow['loss_eur']
_borrow_loss_pct = _worst_borrow['loss_pct']

print(f"Counterparty Stress — AIFM Private Debt  |  NAV: EUR {NAV:,.0f}")
print(f"Senior secured recovery: {_RECOVERY_RATE*100:.0f}%  (LGD {(1-_RECOVERY_RATE)*100:.0f}%)\n")
print(f"{'Rank':<6} {'Issuer/Borrower':<28} {'Exposure (EUR)':>16} {'% NAV':>8} {'Loss (EUR)':>14} {'% NAV':>8}")
print('─' * 84)
for rank, (_, r) in enumerate(_by_borrower.head(5).iterrows(), 1):
    print(f"{rank:<6} {str(r['issuer']):<28} {r['exposure_eur']:>15,.0f} "
          f"{r['exp_pct']*100:>7.1f}%  {r['loss_eur']:>13,.0f} {r['loss_pct']*100:>7.1f}%")
print('─' * 84)
_limit_breach = _worst_borrow['exp_pct'] > 0.20
print(f"\nWorst-case: {_worst_borrow['issuer']} defaults")
print(f"  Exposure   : EUR {_worst_borrow['exposure_eur']:,.0f}  ({_worst_borrow['exp_pct']*100:.1f}% NAV)")
print(f"  Net loss   : EUR {_borrow_loss_eur:,.0f}  ({_borrow_loss_pct*100:.1f}% NAV)")
print(f"  Single-borrower limit (20% NAV): {"⚠ BREACH" if _limit_breach else "✓ Within limit"}")


### 6.5 Combined Stress

The combined scenario joins a credit spread shock with a redemption demand.

The output shows how valuation losses and liquidity pressure interact in a private debt portfolio.

In [ ]:
# MRS-35: Combined stress — credit spread +150bps AND 25% redemption simultaneously
# cr is already in scope from Section 4 stress cell
_comb_mkt_eur_pd = cr['stressed_pnl_eur']
_comb_nav_st_pd  = NAV + _comb_mkt_eur_pd

# Under credit stress, liquid assets shrink proportionally with spread widening
_base_red_pd         = _redstress[0.25]
_comb_liquid_st_pd   = _base_red_pd['liquid_assets_eur'] * (1 - 0.10)  # 10% haircut on liquid book
_comb_redeem_eur_pd  = NAV * 0.25
_comb_gap_st_pd      = _comb_liquid_st_pd - _comb_redeem_eur_pd
_comb_cov_st_pd      = _comb_liquid_st_pd / _comb_redeem_eur_pd if _comb_redeem_eur_pd > 0 else float('inf')
_comb_action_pd      = 'Can meet redemption' if _comb_gap_st_pd >= 0 else 'Gate / partial suspension required'

print(f"Combined Stress — AIFM Private Debt  |  Credit +150bps + 25% Redemption")
print(f"Baseline NAV: EUR {NAV/1e6:,.1f}M\n")
print(f"  Market shock (credit spread +150bps):")
print(f"    Portfolio P&L  : EUR {_comb_mkt_eur_pd/1e6:,.1f}M  ({_comb_mkt_eur_pd/NAV*100:.1f}% NAV)")
print(f"    Stressed NAV   : EUR {_comb_nav_st_pd/1e6:,.1f}M")
print()
print(f"  Liquidity impact (25% redemption, liquid assets -10% haircut):")
print(f"    Redemption     : EUR {_comb_redeem_eur_pd/1e6:,.1f}M  (25% pre-stress NAV)")
print(f"    Liquid assets  : EUR {_comb_liquid_st_pd/1e6:,.1f}M  (post credit haircut)")
print(f"    Liquidity gap  : EUR {_comb_gap_st_pd/1e6:,.1f}M  |  Coverage: {_comb_cov_st_pd:.2f}x")
print(f"    Action         : {_comb_action_pd}")
print()
_total_stress_pd = _comb_mkt_eur_pd - max(0.0, -_comb_gap_st_pd)
_total_pct_pd    = _total_stress_pd / NAV * 100
print(f"  Total combined impact on NAV: EUR {_total_stress_pd/1e6:,.1f}M  ({_total_pct_pd:.1f}% of NAV)")
print(f"  Regulatory note: ESMA/2020/1498 §48 — combined stress is a mandatory Annex VI scenario")


## ESG Risk Indicators

The notebook computes portfolio-level ESG indicators using NAV-weighted averages and the fund's internal ESG threshold.

The output summarises ESG scores, low-score exposure, controversy flags, and carbon intensity.

In [ ]:
esg_df  = esg_u.build_esg_df(risk_df, BBG, ENGINE, FUND_ID, VALUATION_DATE)
esg_u.display_esg_assets(esg_df)

In [ ]:
esg_u.display_esg_summary(esg_df)

---

## 7. Annex IV Reporting Context

This section summarises the private debt indicators used as Annex IV-style reporting inputs: leverage, liquidity profile, investor concentration, borrower default stress, credit spread stress, and ESG indicators.

For this private debt AIF, the relevant monitoring lens is credit and liquidity-based rather than a generic VaR framework.

In [ ]:
from src.reporting.annex_iv import build_annex_iv, export_annex_iv_excel

ANNEX_IV_QUARTER = '2026-03-31'

annex_iv = build_annex_iv(ENGINE, FUND_ID, quarter=ANNEX_IV_QUARTER)
print(f"Annex IV — {FUND_ID}   Reporting period: Q1 2026 ({ANNEX_IV_QUARTER})")
print(f"NAV: EUR {annex_iv['_nav']/1e6:,.1f}M")

In [ ]:
def get_row_index(df, sections, colname):
    mask = df[colname].str.upper().isin([s.upper() for s in sections])
    section_rows = df[mask].index.tolist()
    return section_rows

sections_dict = {
    'identification':('field',['FUND IDENTITY', 'AIFM', 'COUNTERPARTIES', 
        'REPORTING', 'REDEMPTION TERMS', 'LEVERAGE LIMITS (RMP)']), 
    'breakdown':('Category',['Asset Class', 'Geography', 'Currency', 'Top 5 positions']),
    'risk_measures':('field', [f'VaR & ES (99% confidence, historical simulation, 250 days)', 'LEVERAGE', 'LIQUIDITY HEADLINE']
),
    'leverage_detail':('item',[
        f'GROSS METHOD — breakdown by source (EU231/2013 Art. 7)',
        f'COMMITMENT METHOD (EU231/2013 Art. 8)']),	
    'liquidity_buckets':None,
    'liquidity_terms':('field',['INVESTOR CONCENTRATION (ESMA thresholds)']),
    # '_nav':[],
 }

col_widths_bkdn = {
    'Category':'100px', 
    'NAV (EUR)':'200px', 
    'NAV %':'100px'
    }

def annex_iv_section(annex_iv, section, col_widths=None, fmt=None):
    
    df = annex_iv[section]
    df.columns = df.columns.str.replace('PCT', '%')
    col_align_override = {col:'right' for col in df.columns}
    
    section_rows = None
    if sections_dict[section]:
        colname, sub_sections = sections_dict[section]
        section_rows = get_row_index(df, sub_sections, colname)

    phtml.display_dark_table(
        df,
        caption=section.upper().replace('_',' '),
        fmt = fmt,
        # col_styles          : dict | None = None,
        col_align_override = col_align_override,
        highlight_rows = section_rows,
        col_widths=col_widths,
    )

In [ ]:
annex_iv_section(annex_iv, 'identification')

In [ ]:
print("\n=== Section 2.1 — Asset Class Breakdown ===")
display(annex_iv['asset_class_breakdown'])

print("\n=== Section 2.2 — Geographic Breakdown ===")
display(annex_iv['geography_breakdown'])

print("\n=== Section 2.4 — Top 5 Positions ===")
display(annex_iv['top5_positions'])


In [ ]:
print("\n=== Section 3 — Risk Measures ===")
display(annex_iv['risk_measures'])

print("\n=== Section 4 — Leverage Detail ===")
display(annex_iv['leverage_detail'])

print("\n=== Section 5 — Liquidity Profile ===")
liq = annex_iv['liquidity_buckets']
cols = [c for c in ['bucket', 'nav_eur', 'nav_pct', 'cumulative_pct'] if c in liq.columns]
display(liq[cols])

print("\n=== Section 5 — Redemption Terms & Investor Concentration ===")
display(annex_iv['liquidity_terms'])
